# Clustering from Detchar Group

This code need to be run in CASCINA cluster from Italy, because the data we will use comes from the LVK colaboration.

In [ ]:
# Importação
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
warnings.filterwarnings("ignore", message="Pandas requires version")

import os
import ast
import numpy as np
import pandas as pd
from glob import glob
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from datetime import datetime, timedelta
from gwpy.time import to_gps
from gwpy.table import EventTable
from astropy.table import unique as at_unique
from astropy.units import Unit, dimensionless_unscaled
from gwpy.timeseries import TimeSeries
from gwpy.segments import DataQualityFlag
from virgotools import getChannel

In [ ]:
#Encontrar os dados
def find_trigger_files(ch, start, end, trigpath="/data/*/omicron/", ext="root"):
    allfiles = glob.glob(os.path.join(trigpath, "{}/{}_OMICRON".format(*ch.split(":")), "**", f"*.{ext}"), recursive=True)
    if not allfiles:
        print(f"Nenhum arquivo encontrado no caminho {trigpath}")
        return []
    allstart, alldur = np.array([f.rstrip(f".{ext}").split("-")[-2:] for f in allfiles]).astype(int).T
    try:
        rightfiles = np.asarray(allfiles)[(allstart + alldur > start) & (end > allstart)]
    except TypeError:
        print(f"Nenhum arquivo relevante encontrado entre {start} e {end}")
        return []
    return rightfiles

#Manipulação de String I
def FrV_to_TS(frV):
    try:
        unit = Unit(frV.unit)
    except Exception:
        unit = dimensionless_unscaled
    return TimeSeries(data=frV.data, t0=frV.gps, sample_rate=frV.fsample, unit=unit, channel=frV.name)

#Manipulação de String II
def read2TS(*args, **kwargs):
    with getChannel(*args, **kwargs) as frv:
        return FrV_to_TS(frv)

In [ ]:
#Processo de clusterização
def clusterize_triggers(table, window=0.1):
    if len(table) == 0:
        return EventTable(rows=[])  # retorna tabela vazia se não houver triggers

    table = table.copy()
    table.sort('tstart')

    cluster_ids = np.zeros(len(table), dtype=int)
    cluster_index = 0
    for i in range(1, len(table)):
        delta_t = table[i]['tstart'] - table[i-1]['tend']
        if delta_t <= window:
            cluster_ids[i] = cluster_index
        else:
            cluster_index += 1
            cluster_ids[i] = cluster_index

    table['cluster'] = cluster_ids
    summary_rows = []
    for cl in np.unique(cluster_ids):
        cluster_rows = table[table['cluster'] == cl]
        if len(cluster_rows) == 0:
            continue
        max_idx = np.argmax(cluster_rows['snr'])
        max_row = cluster_rows[max_idx]

        summary_rows.append({
            'cluster': cl,
            'tstart': np.min(cluster_rows['tstart']),
            'tend': np.max(cluster_rows['tend']),
            'fmin': np.min(cluster_rows['fstart']),
            'fmax': np.max(cluster_rows['fend']),
            'time': max_row['time'],
            'frequency': max_row['frequency'],
            'q': max_row['q'],
            'snr': max_row['snr'],
            'phase': max_row['phase']
        })
    return EventTable(rows=summary_rows)

In [ ]:
# Função para calcular tempo ativo
def dq_active_seconds(chDQ, state_val, start_dt, end_dt, label="Low Noise"):
    """Retorna (total_active_seconds, DQflag) no intervalo [start_dt, end_dt]."""
    start = to_gps(start_dt).gpsSeconds if hasattr(to_gps(start_dt), "gpsSeconds") else to_gps(start_dt)
    end   = to_gps(end_dt).gpsSeconds   if hasattr(to_gps(end_dt), "gpsSeconds")   else to_gps(end_dt)

    DQTS = read2TS("trend", chDQ, start, end)
    sTS = DQTS >= state_val
    DQ = sTS.to_dqflag(label=label)

    total_active = float(sum(seg[1] - seg[0] for seg in DQ.active))  # segundos
    return total_active, DQ

In [ ]:
#Canais 
ch = "V1:Hrec_hoft_16384Hz"
chDQ = "V1:META_ITF_LOCK_index"
state_val = 135  # low-noise

#Janela de operação
run_start_day = datetime(2024, 5, 10, 0, 0, 0)
run_end_day   = datetime(2024, 5, 20, 23, 59, 59)


snr_thresholds = [6.5, 10] #SNR_cut
cluster_window = 0.1  # Janela de clusterização (segundos)

days_per_part = 20
part_number = 1
current_day = run_start_day
monthly_results = []

# ---------------- Loop principal ----------------
while current_day <= run_end_day:
    day_start = datetime(current_day.year, current_day.month, current_day.day, 0, 0, 0)
    day_end   = datetime(current_day.year, current_day.month, current_day.day, 23, 59, 59)
    print(f"Processando dia: {current_day.date()}")

    # --- tempo ativo ---
    try:
        total_active_seconds, DQ = dq_active_seconds(chDQ, state_val, day_start, day_end)
    except Exception as e:
        print(f"  Falha ao obter DQ ({current_day.date()}): {e}")
        total_active_seconds = 0.0
    total_active_hours = total_active_seconds / 3600.0

    # --- triggers ---
    gps_start = to_gps(day_start)
    gps_end   = to_gps(day_end)
    trigfiles = find_trigger_files(ch, gps_start, gps_end)
    if len(trigfiles) == 0:
        print(f"Nenhum arquivo encontrado para o dia {day_start.date()}")
        gltab = EventTable(rows=[])
    else:
        gltab = EventTable.read([trigfiles[0]], treename='triggers;1', nproc=3)
        gltab.sort("time")
        if len(gltab) > 0:
            gltab = at_unique(gltab)
            
    print(f'{gltab}\n\n')

    # --- clusterização ---
    clustered = clusterize_triggers(gltab, window=cluster_window) if len(gltab) > 0 else EventTable(rows=[])

    # --- counts pós SNR cut ---
    counts_nonclustered = {snr: int(np.sum(gltab['snr'] > snr)) if len(gltab) > 0 else 0 for snr in snr_thresholds}
    counts_clustered   = {snr: int(np.sum(clustered['snr'] > snr)) if len(clustered) > 0 else 0 for snr in snr_thresholds}

    monthly_results.append({
        "date": current_day.date(),
        "n_triggers_nonclustered": int(len(gltab)),
        "n_triggers_clustered": int(len(clustered)),
        "counts_nonclustered": counts_nonclustered,
        "counts_clustered": counts_clustered,
        "total_active_seconds": total_active_seconds
    })

    # --- salvar parcial ---
    if len(monthly_results) % days_per_part == 0:
        fname = f"O4b_part{part_number}.npy"
        np.save(fname, monthly_results)
        print(f"=== Resultados parciais salvos em {fname} ===\n")
        part_number += 1
        monthly_results = []

    current_day += timedelta(days=1)

# --- salvar última parte ---
if len(monthly_results) > 0:
    fname = f"O4b_part{part_number}.npy"
    np.save(fname, monthly_results)
    print(f"=== Última parte salva em {fname} ===\n")

# ---------------- Combinar todas as partes em CSV final ----------------
all_parts = []
part_number = 1
while True:
    fname = f"O4b_part{part_number}.npy"
    if not os.path.exists(fname):
        break
    arr = np.load(fname, allow_pickle=True)
    all_parts.extend(arr)
    part_number += 1

# converter para DataFrame
df = pd.DataFrame(all_parts)

# remover linhas com counts zero
def nonzero_counts(row):
    nc = row.get('counts_nonclustered', {}) if isinstance(row.get('counts_nonclustered', {}), dict) else {}
    c  = row.get('counts_clustered', {}) if isinstance(row.get('counts_clustered', {}), dict) else {}
    return any(v > 0 for v in nc.values()) or any(v > 0 for v in c.values())

df = df[df.apply(nonzero_counts, axis=1)]

# salvar CSV final
df.to_csv("O4b_full.csv", index=False)
print("Arquivo final salvo: O4b_full.csv")

In [ ]:
def process_file2(filename):
    df = pd.read_csv(filename)
    
    # filtro 2: remover total_active_seconds nulos ou iguais a 0
    df = df[df['total_active_seconds'].notnull() & (df['total_active_seconds'] > 0)]

    # sobrescreve o CSV
    new_name = filename.replace(".csv", "_clean.csv")
    df.to_csv(new_name, index=False)
    print(f"{new_name} salvo com {len(df)} linhas")

# processar os dois arquivos
'''process_file2("O4a_full.csv")'''
process_file2("O4b_full.csv")

In [ ]:
#Exibir resultado
def load_and_sum(filename):
    df = pd.read_csv(filename)

    # Converte os dicionários salvos como string em dict
    def safe_eval(x):
        if isinstance(x, str):
            try:
                return ast.literal_eval(x)
            except Exception:
                return {}
        return x if isinstance(x, dict) else {}

    df['counts_nonclustered'] = df['counts_nonclustered'].apply(safe_eval)
    df['counts_clustered']    = df['counts_clustered'].apply(safe_eval)

    totals = {
        "Total_Counts_NonClusted_6.5": df['counts_nonclustered'].apply(lambda d: d.get(6.5, 0)).sum(),
        "Total_Counts_NonClusted_10": df['counts_nonclustered'].apply(lambda d: d.get(10, 0)).sum(),
        "Total_Counts_Clusted_6.5": df['counts_clustered'].apply(lambda d: d.get(6.5, 0)).sum(),
        "Total_Counts_Clusted_10": df['counts_clustered'].apply(lambda d: d.get(10, 0)).sum(),
        "Total_Time": df['total_active_seconds'].sum() / 3600
    }

    return totals

# aplica nos dois arquivos
'''res_a = load_and_sum("O4a_full_clean.csv")'''
res_b = load_and_sum("O4b_full_clean.csv")

'''print("=== O4a ===")
for k, v in res_a.items():
    print(f"{k}: {v}")'''

print("\n=== O4b ===")
for k, v in res_b.items():
    print(f"{k}: {v}")

In [ ]:
# -------------------------
# Função de leitura
# -------------------------
def load_and_sum(filename):
    df = pd.read_csv(filename)

    # Converte os dicionários salvos como string em dict
    def safe_eval(x):
        if isinstance(x, str):
            try:
                return ast.literal_eval(x)
            except Exception:
                return {}
        return x if isinstance(x, dict) else {}

    df['counts_nonclustered'] = df['counts_nonclustered'].apply(safe_eval)
    df['counts_clustered']    = df['counts_clustered'].apply(safe_eval)

    totals = {
        "Total_Counts_NonClustered_6.5": df['counts_nonclustered'].apply(lambda d: d.get(6.5, 0)).sum(),
        "Total_Counts_NonClustered_10": df['counts_nonclustered'].apply(lambda d: d.get(10, 0)).sum(),
        "Total_Counts_Clustered_6.5": df['counts_clustered'].apply(lambda d: d.get(6.5, 0)).sum(),
        "Total_Counts_Clustered_10": df['counts_clustered'].apply(lambda d: d.get(10, 0)).sum(),
        "Total_Time": df['total_active_seconds'].sum() / 3600  # converte para horas
    }
    return totals

In [ ]:
result = load_and_sum('O4b_full_clean.csv')
result

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# Ajuste global de estilo e fonte tipo LaTeX
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Computer Modern'],
    'text.usetex': True,
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'legend.fontsize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12
})

# Dados
detectors = ['Virgo', 'LHO', 'LLO']

totals = [119.0, 77.2, 53.7]
cuts   = [21.3, 5.9, 30.8]

colors_dark = ['#087F23', '#fa8072', '#1e90ff']   # SNR > 10
colors_light = ['#A8E6A1', '#fcbfb8', '#8ec7ff']  # 6.5 < SNR < 10

fig, ax = plt.subplots(figsize=(5, 6))

bar_width = 0.5
x = np.arange(len(detectors))

for i, (total, cut, c_dark, c_light) in enumerate(zip(totals, cuts, colors_dark, colors_light)):
    # Parte escura (SNR > 10)
    ax.bar(x[i], cut, width=bar_width, color=c_dark, edgecolor='black')
    ax.text(x[i], cut/2, f'{cut:.1f}', ha='center', va='center', 
            color='white', fontsize=12, fontweight='bold')
    
    # Parte clara (6.5 < SNR < 10)
    remainder = total - cut
    ax.bar(x[i], remainder, bottom=cut, width=bar_width, color=c_light, edgecolor='black')
    ax.text(x[i], cut + remainder/2, f'{total:.1f}', ha='center', va='center',
            color='black', fontsize=12, fontweight='bold')

# Legenda
legend_patches = [
    Patch(color='#1e90ff', label=r'SNR$>10$'),
    Patch(color='#8ec7ff', label=r'SNR$>6.5$'),
    Patch(color='#fa8072', label=r'SNR$>10$'),
    Patch(color='#fcbfb8', label=r'SNR$>6.5$'),
    Patch(color='#087F23', label=r'SNR$>10$'),
    Patch(color='#A8E6A1', label=r'SNR$>6.5$')
]
ax.legend(handles=legend_patches, title='Faixas de SNR', bbox_to_anchor=(1.05, 1), loc='upper left')

ax.set_xticks(x)
ax.set_xticklabels(detectors)
ax.set_xlabel('Detectors')
ax.set_ylabel('Glitches rate per hour')
'''axs[0].set_title("O4a")'''
plt.grid(None)
plt.tight_layout()
'''plt.savefig('glitches_rate_per_hour_4a.pdf')'''
plt.show()
